# SEC EDGAR Ingestion

Ingests **Company Facts** (XBRL financial concepts) and **Submissions** (recent filing metadata) from the SEC EDGAR REST API for a single company (currently Coinbase Global, Inc., CIK `0001679788`) into two Delta bronze tables: `workspace.bronze.bronze_sec_companyfacts` and `workspace.bronze.bronze_sec_submissions`.

The pipeline fetches JSON from `data.sec.gov`, flattens nested XBRL data into tabular rows in Python, converts to PySpark DataFrames with explicit schemas, applies date parsing and deduplication, and appends to Delta. Uses `mode("append")` — re-runs will accumulate duplicates unless the table is cleared first.

In [0]:
import requests

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType
)
from pyspark.sql.functions import (
    current_timestamp,
    try_to_date,
    col
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

spark = SparkSession.builder.getOrCreate()

# Coinbase Global, Inc.
CIK = "0001679788"

# Replace with your actual email
USER_AGENT = "Kyle McClelland your_email@example.com"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate"
}

COMPANY_FACTS_URL = (
    f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
)

SUBMISSIONS_URL = (
    f"https://data.sec.gov/submissions/CIK{CIK}.json"
)

COMPANY_FACTS_TABLE = "bronze.bronze_sec_companyfacts"
SUBMISSIONS_TABLE = "bronze.bronze_sec_submissions"


# ============================================================
# 2. SEC REQUEST FUNCTION
# ============================================================

def get_sec_json(url):

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return response.json()


# ============================================================
# 3. DOWNLOAD COMPANY FACTS
# ============================================================

company_facts_data = get_sec_json(
    COMPANY_FACTS_URL
)

print(
    f"Downloaded Company Facts for: "
    f"{company_facts_data.get('entityName')}"
)


# ============================================================
# 4. FLATTEN COMPANY FACTS
# ============================================================

company_fact_rows = []

facts = company_facts_data.get(
    "facts",
    {}
)

for taxonomy, taxonomy_facts in facts.items():

    for concept_name, concept_data in taxonomy_facts.items():

        label = concept_data.get("label")

        description = concept_data.get(
            "description"
        )

        units = concept_data.get(
            "units",
            {}
        )

        for unit, observations in units.items():

            for observation in observations:

                raw_value = observation.get(
                    "val"
                )

                try:
                    numeric_value = float(
                        raw_value
                    )

                except (TypeError, ValueError):
                    numeric_value = None

                company_fact_rows.append({

                    "cik":
                        int(company_facts_data["cik"]),

                    "company_name":
                        company_facts_data.get(
                            "entityName"
                        ),

                    "taxonomy":
                        taxonomy,

                    "concept":
                        concept_name,

                    "label":
                        label,

                    "description":
                        description,

                    "unit":
                        unit,

                    "value":
                        numeric_value,

                    "start_date":
                        observation.get("start"),

                    "end_date":
                        observation.get("end"),

                    "filing_date":
                        observation.get("filed"),

                    "form":
                        observation.get("form"),

                    "fiscal_year":
                        observation.get("fy"),

                    "fiscal_period":
                        observation.get("fp"),

                    "frame":
                        observation.get("frame"),

                    "accession_number":
                        observation.get("accn")
                })


print(
    f"Flattened "
    f"{len(company_fact_rows):,} "
    f"Company Fact observations."
)


# ============================================================
# 5. COMPANY FACTS SCHEMA
# ============================================================

company_facts_schema = StructType([

    StructField(
        "cik",
        LongType(),
        True
    ),

    StructField(
        "company_name",
        StringType(),
        True
    ),

    StructField(
        "taxonomy",
        StringType(),
        True
    ),

    StructField(
        "concept",
        StringType(),
        True
    ),

    StructField(
        "label",
        StringType(),
        True
    ),

    StructField(
        "description",
        StringType(),
        True
    ),

    StructField(
        "unit",
        StringType(),
        True
    ),

    StructField(
        "value",
        DoubleType(),
        True
    ),

    StructField(
        "start_date",
        StringType(),
        True
    ),

    StructField(
        "end_date",
        StringType(),
        True
    ),

    StructField(
        "filing_date",
        StringType(),
        True
    ),

    StructField(
        "form",
        StringType(),
        True
    ),

    StructField(
        "fiscal_year",
        IntegerType(),
        True
    ),

    StructField(
        "fiscal_period",
        StringType(),
        True
    ),

    StructField(
        "frame",
        StringType(),
        True
    ),

    StructField(
        "accession_number",
        StringType(),
        True
    )
])


# ============================================================
# 6. CREATE COMPANY FACTS DATAFRAME
# ============================================================

company_facts_df = spark.createDataFrame(
    company_fact_rows,
    schema=company_facts_schema
)

company_facts_df = (
    company_facts_df

    .withColumn(
        "start_date",
        try_to_date(
            col("start_date")
        )
    )

    .withColumn(
        "end_date",
        try_to_date(
            col("end_date")
        )
    )

    .withColumn(
        "filing_date",
        try_to_date(
            col("filing_date")
        )
    )

    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)


# ============================================================
# 7. DOWNLOAD SEC SUBMISSIONS
# ============================================================

submissions_data = get_sec_json(
    SUBMISSIONS_URL
)

print(
    f"Downloaded submissions for: "
    f"{submissions_data.get('name')}"
)


# ============================================================
# 8. COMPANY METADATA
# ============================================================

company_name = submissions_data.get(
    "name"
)

sic = submissions_data.get(
    "sic"
)

sic_description = submissions_data.get(
    "sicDescription"
)

tickers = submissions_data.get(
    "tickers",
    []
)

exchanges = submissions_data.get(
    "exchanges",
    []
)

ticker = (
    tickers[0]
    if tickers
    else None
)

exchange = (
    exchanges[0]
    if exchanges
    else None
)


# ============================================================
# 9. FLATTEN RECENT FILINGS
# ============================================================

recent_filings = (
    submissions_data
    .get("filings", {})
    .get("recent", {})
)

accession_numbers = recent_filings.get(
    "accessionNumber",
    []
)

filing_dates = recent_filings.get(
    "filingDate",
    []
)

report_dates = recent_filings.get(
    "reportDate",
    []
)

forms = recent_filings.get(
    "form",
    []
)

primary_documents = recent_filings.get(
    "primaryDocument",
    []
)

primary_document_descriptions = recent_filings.get(
    "primaryDocDescription",
    []
)


submission_rows = []

for i in range(
    len(accession_numbers)
):

    submission_rows.append({

        "cik":
            int(CIK),

        "company_name":
            company_name,

        "ticker":
            ticker,

        "exchange":
            exchange,

        "sic":
            sic,

        "sic_description":
            sic_description,

        "accession_number":
            accession_numbers[i],

        "filing_date":
            (
                filing_dates[i]
                if i < len(filing_dates)
                else None
            ),

        "report_date":
            (
                report_dates[i]
                if i < len(report_dates)
                else None
            ),

        "form":
            (
                forms[i]
                if i < len(forms)
                else None
            ),

        "primary_document":
            (
                primary_documents[i]
                if i < len(
                    primary_documents
                )
                else None
            ),

        "primary_document_description":
            (
                primary_document_descriptions[i]
                if i < len(
                    primary_document_descriptions
                )
                else None
            )
    })


print(
    f"Flattened "
    f"{len(submission_rows):,} "
    f"SEC filing records."
)


# ============================================================
# 10. SUBMISSIONS SCHEMA
# ============================================================

submissions_schema = StructType([

    StructField(
        "cik",
        LongType(),
        True
    ),

    StructField(
        "company_name",
        StringType(),
        True
    ),

    StructField(
        "ticker",
        StringType(),
        True
    ),

    StructField(
        "exchange",
        StringType(),
        True
    ),

    StructField(
        "sic",
        StringType(),
        True
    ),

    StructField(
        "sic_description",
        StringType(),
        True
    ),

    StructField(
        "accession_number",
        StringType(),
        True
    ),

    StructField(
        "filing_date",
        StringType(),
        True
    ),

    StructField(
        "report_date",
        StringType(),
        True
    ),

    StructField(
        "form",
        StringType(),
        True
    ),

    StructField(
        "primary_document",
        StringType(),
        True
    ),

    StructField(
        "primary_document_description",
        StringType(),
        True
    )
])


# ============================================================
# 11. CREATE SUBMISSIONS DATAFRAME
# ============================================================

submissions_df = spark.createDataFrame(
    submission_rows,
    schema=submissions_schema
)

submissions_df = (
    submissions_df

    .withColumn(
        "filing_date",
        try_to_date(
            col("filing_date")
        )
    )

    .withColumn(
        "report_date",
        try_to_date(
            col("report_date")
        )
    )

    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)


# ============================================================
# 12. REMOVE DUPLICATES FROM CURRENT LOAD
# ============================================================

company_facts_df = (
    company_facts_df
    .dropDuplicates([
        "cik",
        "taxonomy",
        "concept",
        "unit",
        "end_date",
        "accession_number"
    ])
)

submissions_df = (
    submissions_df
    .dropDuplicates([
        "cik",
        "accession_number"
    ])
)


# ============================================================
# 13. DISPLAY DATA BEFORE LOAD
# ============================================================

print("\nSEC COMPANY FACTS")

display(
    company_facts_df
    .orderBy(
        col("filing_date").desc()
    )
)


print("\nSEC SUBMISSIONS")

display(
    submissions_df
    .orderBy(
        col("filing_date").desc()
    )
)


# ============================================================
# 14. LOAD COMPANY FACTS INTO DELTA / SQL TABLE
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

(
    company_facts_df
    .write
    .format("delta")
    .mode("append")
    .option(
        "mergeSchema",
        "true"
    )
    .saveAsTable(
        COMPANY_FACTS_TABLE
    )
)


# ============================================================
# 15. LOAD SUBMISSIONS INTO DELTA / SQL TABLE
# ============================================================

(
    submissions_df
    .write
    .format("delta")
    .mode("append")
    .option(
        "mergeSchema",
        "true"
    )
    .saveAsTable(
        SUBMISSIONS_TABLE
    )
)


# ============================================================
# 16. VALIDATE SQL TABLE LOAD
# ============================================================

print("\n============================================")
print("SQL / DELTA LOAD COMPLETE")
print("============================================")

print(
    f"Loaded table: "
    f"{COMPANY_FACTS_TABLE}"
)

print(
    f"Loaded table: "
    f"{SUBMISSIONS_TABLE}"
)


# Display the actual SQL tables after load

display(
    spark.sql(
        f"""
        SELECT *
        FROM {COMPANY_FACTS_TABLE}
        ORDER BY filing_date DESC
        """
    )
)

display(
    spark.sql(
        f"""
        SELECT *
        FROM {SUBMISSIONS_TABLE}
        ORDER BY filing_date DESC
        """
    )
)

In [0]:
company_facts_df.select(["concept","value"]).show()

In [0]:
display(company_facts_df.select("concept").distinct())